# ЛР4 — Семантический анализатор: ML.NET + ONNX Runtime (без LSTM)

Три анализатора тональности на базе результатов ЛР3:
- A) ML.NET FeaturizeText + SDCA MaximumEntropy
- B) ML.NET FeaturizeText + L-BFGS MaximumEntropy
- C) CNN (ONNX Runtime, обучение в Python, инференс в C#)

Заметка: LSTM/seq2seq не используются. CNN обучается отдельно и экспортируется в ONNX.


In [ ]:
#r "nuget: Microsoft.ML, 3.0.1"
#r "nuget: Microsoft.ML.FastTree, 3.0.1"
#r "nuget: Microsoft.ML.OnnxRuntime, 1.19.0"
#r "nuget: CsvHelper, 30.0.1"


In [ ]:
using System;
using System.IO;
using System.Linq;
using System.Text;
using System.Text.RegularExpressions;
using System.Collections.Generic;
using System.Globalization;
using CsvHelper;
using CsvHelper.Configuration;
using Microsoft.ML;
using Microsoft.ML.Data;
using Microsoft.ML.Transforms.Text;
using Microsoft.ML.Trainers;
using Microsoft.ML.OnnxRuntime;
using Microsoft.ML.OnnxRuntime.Tensors;


In [ ]:
public class TrainRow { public string textID {get;set;} public string text {get;set;} public string selected_text {get;set;} public string sentiment {get;set;} }
public class TestRow  { public string textID {get;set;} public string text {get;set;} public string sentiment {get;set;} }
public class SentimentInput { [LoadColumn(0)] public string Text {get;set;} [LoadColumn(1)] public string Sentiment {get;set;} }
public class SentimentPrediction { [ColumnName("PredictedLabel")] public string PredictedLabel {get;set;} }

static string Normalize(string s){ if (string.IsNullOrWhiteSpace(s)) return string.Empty; s=s.ToLowerInvariant(); s=Regex.Replace(s, "https?://\\S+", " " ); s=Regex.Replace(s, "[\\\"'`]+", "" ); s=Regex.Replace(s, "[^\\p{L}\\p{Nd}#@ ]+", " " ); s=Regex.Replace(s, "\\s+", " " ).Trim(); return s; }

static List<SentimentInput> LoadTrain(string path){ using var sr=new StreamReader(path, Encoding.UTF8); var cfg=new CsvConfiguration(CultureInfo.InvariantCulture){HasHeaderRecord=true, DetectDelimiter=true, IgnoreBlankLines=true, BadDataFound=null}; using var csv=new CsvReader(sr,cfg); var list=new List<SentimentInput>(); foreach (var r in csv.GetRecords<TrainRow>()){ if (string.IsNullOrWhiteSpace(r.text) || string.IsNullOrWhiteSpace(r.sentiment)) continue; list.Add(new SentimentInput{ Text=Normalize(r.text), Sentiment=r.sentiment.Trim()}); } return list; }
static List<SentimentInput> LoadTest(string path){ using var sr=new StreamReader(path, Encoding.UTF8); var cfg=new CsvConfiguration(CultureInfo.InvariantCulture){HasHeaderRecord=true, DetectDelimiter=true, IgnoreBlankLines=true, BadDataFound=null}; using var csv=new CsvReader(sr,cfg); var list=new List<SentimentInput>(); foreach (var r in csv.GetRecords<TestRow>()){ if (string.IsNullOrWhiteSpace(r.text) || string.IsNullOrWhiteSpace(r.sentiment)) continue; list.Add(new SentimentInput{ Text=Normalize(r.text), Sentiment=r.sentiment.Trim()}); } return list; }

var dataDir = Path.Combine("..", "data");
var trainPath = Path.Combine(dataDir, "train.csv");
var testPath  = Path.Combine(dataDir, "test.csv");
var train = LoadTrain(trainPath); var test = LoadTest(testPath);
Console.WriteLine($"Train={train.Count}, Test={test.Count}");


In [ ]:
var ml = new MLContext(seed:42);
var trainDV = ml.Data.LoadFromEnumerable(train);
var testDV  = ml.Data.LoadFromEnumerable(test);

IEstimator<ITransformer> Pipeline(string trainer){
    var text = ml.Transforms.Text.FeaturizeText("Features", nameof(SentimentInput.Text));
    var key = ml.Transforms.Conversion.MapValueToKey("Label", nameof(SentimentInput.Sentiment));
    IEstimator<ITransformer> tr = trainer=="sdca" ? ml.MulticlassClassification.Trainers.SdcaMaximumEntropy(labelColumnName:"Label", featureColumnName:"Features") : ml.MulticlassClassification.Trainers.LbfgsMaximumEntropy(labelColumnName:"Label", featureColumnName:"Features");
    return text.Append(key).Append(tr).Append(ml.Transforms.Conversion.MapKeyToValue("PredictedLabel"));
}

(ITransformer model, string name) TrainEval(string name, string trainer){
    var pipe = Pipeline(trainer);
    var model = pipe.Fit(trainDV);
    var preds = model.Transform(testDV);
    var metrics = ml.MulticlassClassification.Evaluate(preds, labelColumnName:"Label", predictedLabelColumnName:"PredictedLabel");
    Console.WriteLine($"[{name}] MicroAcc={metrics.MicroAccuracy:F3}  MacroAcc={metrics.MacroAccuracy:F3}");
    return (model,name);
}

var (modelA, nameA) = TrainEval("A: FeaturizeText + SDCA", "sdca");
var (modelB, nameB) = TrainEval("B: FeaturizeText + L-BFGS", "lbfgs");


In [ ]:
string modelsDir = Path.Combine("..", "models");
string onnxPath  = Path.Combine(modelsDir, "cnn_text.onnx");
string vocabPath = Path.Combine(modelsDir, "vocab.json");
string cfgPath   = Path.Combine(modelsDir, "config.json");

Dictionary<string,int>? vocab = null; int maxLen = 40;
if (File.Exists(vocabPath)) { var json = System.Text.Json.JsonDocument.Parse(File.ReadAllText(vocabPath)); vocab = new Dictionary<string,int>(); foreach (var kv in json.RootElement.EnumerateObject()) vocab[kv.Name]=kv.Value.GetInt32(); }
if (File.Exists(cfgPath)) { var json = System.Text.Json.JsonDocument.Parse(File.ReadAllText(cfgPath)); if (json.RootElement.TryGetProperty("max_len", out var v)) maxLen = v.GetInt32(); }

int Id(string tok){ if (vocab==null) return 1; return vocab.TryGetValue(tok, out var id) ? id : 1; } // 0=pad, 1=unk
int[] ToIds(string text){ var toks = Regex.Split(Normalize(text), "\\s+").Where(t=>t.Length>0).ToArray(); var arr = new int[maxLen]; for (int i=0;i<Math.Min(maxLen,toks.Length);i++) arr[i]=Id(toks[i]); return arr; }

if (File.Exists(onnxPath) && vocab!=null) {
    using var session = new InferenceSession(onnxPath, SessionOptions.MakeSessionOptionWithCudaProvider() ?? new SessionOptions());
    double SoftmaxMax(double[] z, out int arg){ double m=z.Max(); double s=0; var p=new double[z.Length]; for(int i=0;i<z.Length;i++){ p[i]=Math.Exp(z[i]-m); s+=p[i]; } double best=-1; arg=0; for(int i=0;i<z.Length;i++){ var v=p[i]/s; if(v>best){best=v; arg=i;} } return p[arg]/s; }
    int correct=0; int total=0; foreach(var ex in test){ var ids = ToIds(ex.Text); var t = new DenseTensor<long>(new[] {1, maxLen}); for(int i=0;i<maxLen;i++) t[0,i] = ids[i]; var inputs = new List<NamedOnnxValue>{ NamedOnnxValue.CreateFromTensor("input_ids", t) }; using var results = session.Run(inputs); var logits = results.First().AsEnumerable<float>().ToArray(); int arg; var _ = SoftmaxMax(Array.ConvertAll(logits, x=>(double)x), out arg); string pred = arg==0?"negative": arg==1?"neutral":"positive"; if (pred.Equals(ex.Sentiment, StringComparison.OrdinalIgnoreCase)) correct++; total++; }
    Console.WriteLine($"[C: CNN-ONNX] Acc={correct/(double)total:F3} (total={total})");
} else {
    Console.WriteLine("[C: CNN-ONNX] Модель не найдена. Запустите src/train_cnn.py или src/train_cnn.ipynb для обучения и экспорта.");
}


In [ ]:
var samples = new[]{
    "Absolutely love the latest update, everything works flawlessly and makes me so happy!",
    "Customer support was terrible: half of my order was missing and nobody apologized.",
    "It's fine I guess overall, not amazing but acceptable for everyday use."
};

string PredictWith(ITransformer model, string text){
    var engine = ml.Model.CreatePredictionEngine<SentimentInput, SentimentPrediction>(model);
    var input = new SentimentInput{ Text=Normalize(text), Sentiment="neutral" };
    var pred = engine.Predict(input);
    return pred.PredictedLabel;
}

foreach (var s in samples){
    var pa = PredictWith(modelA, s);
    var pb = PredictWith(modelB, s);
    Console.WriteLine(s);
    Console.WriteLine($" A: {pa}");
    Console.WriteLine($" B: {pb}");
}
